# Forecasting PM2.5 at BC Air Quality Stations

This project forecasts fine particulate matter (PM2.5) six hours ahead at 15 air quality monitoring stations in British Columbia, using hourly readings from 2022. PM2.5 is the pollutant behind most air quality advisories in BC, and it spikes sharply during wildfire smoke events.

The models are trained on January to August and tested on September to December. The test period includes the fall 2022 wildfire smoke, so it shows how the models hold up when air quality changes quickly.

This is a solo rebuild of a CPSC4810 team project, focused on the forecasting question the team version left open.

## Data

The data comes from the BC Ministry of Environment and Climate Change Strategy's hourly air quality and meteorological data for 2022, published through the BC Data Catalogue. `merged_df.csv` combines the separate pollutant and weather files into one row per station per hour, for the 15 stations that have a full year of readings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [ ]:
df = pd.read_csv('merged_df.csv', index_col=0)
df['DATE_PST'] = pd.to_datetime(df['DATE_PST'])

# keep one wind direction and one wind speed column, plus the main pollutants and weather
keep = ['DATE_PST', 'STATION_NAME', 'REGION', 'PM25', 'NOx', 'CO',
        'TEMP_MEAN', 'HUMIDITY', 'WSPD_SCLR', 'WDIR_VECT']
df = df[keep].sort_values(['STATION_NAME', 'DATE_PST']).reset_index(drop=True)

print(df.shape)
print()
print(df['STATION_NAME'].value_counts())

## Missing values

PRECIP and PRESSURE were dropped above since most stations don't record them. The remaining columns are mostly complete, and the gradient boosting model can handle the gaps that are left without imputation.

In [ ]:
print(df.isna().mean().round(3).sort_values(ascending=False))

## PM2.5 across the year

Monthly averages show the wildfire smoke in September and October, when PM2.5 rises to two or three times its usual level.

In [ ]:
monthly = df.groupby(df['DATE_PST'].dt.month)['PM25'].agg(['mean', 'max']).round(1)
print(monthly)

daily = df.set_index('DATE_PST').groupby('STATION_NAME')['PM25'].resample('D').mean().unstack(0)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(daily.index, daily.mean(axis=1), color='steelblue')
ax.axvline(pd.Timestamp('2022-09-01'), color='gray', linestyle='--')
ax.text(pd.Timestamp('2022-09-04'), ax.get_ylim()[1] * 0.9, 'test period starts', color='gray')
ax.set_title('Daily average PM2.5 across 15 BC stations, 2022')
ax.set_ylabel('PM2.5 (µg/m³)')
plt.tight_layout()
plt.show()

## Feature engineering

These features are adapted from my feature engineering notebook in the team project. Every lag and rolling value is computed within each station so that one station's readings never leak into another's.

- **Lags** give the PM2.5 reading from 1, 2, 3 and 24 hours earlier
- **Rolling averages** smooth PM2.5 over the last 3 and 24 hours (the 24-hour average is the standard health metric)
- **Cyclical time encoding** turns hour and month into sine and cosine values so that 11pm sits next to midnight
- **Flags** mark rush hour and weekends
- **Weather and other pollutants** come from the same hour

The target is the PM2.5 reading six hours later, set by `HORIZON` below.

In [ ]:
g = df.groupby('STATION_NAME')

for lag in [1, 2, 3, 24]:
    df[f'PM25_lag{lag}'] = g['PM25'].shift(lag)

df['PM25_roll3'] = g['PM25'].transform(lambda x: x.rolling(3, min_periods=2).mean())
df['PM25_roll24'] = g['PM25'].transform(lambda x: x.rolling(24, min_periods=18).mean())
df['NOx_lag1'] = g['NOx'].shift(1)

hour = df['DATE_PST'].dt.hour
month = df['DATE_PST'].dt.month
df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * hour / 24)
df['month_sin'] = np.sin(2 * np.pi * month / 12)
df['month_cos'] = np.cos(2 * np.pi * month / 12)
df['is_rush_hour'] = hour.isin([7, 8, 9, 16, 17, 18]).astype(int)
df['is_weekend'] = df['DATE_PST'].dt.dayofweek.isin([5, 6]).astype(int)

# wind direction is an angle, so split it into east-west and north-south parts
rad = np.deg2rad(df['WDIR_VECT'])
df['wind_u'] = np.sin(rad) * df['WSPD_SCLR']
df['wind_v'] = np.cos(rad) * df['WSPD_SCLR']

# target is PM2.5 at the same station HORIZON hours later
HORIZON = 6
df['target'] = g['PM25'].shift(-HORIZON)

features = ['PM25', 'PM25_lag1', 'PM25_lag2', 'PM25_lag3', 'PM25_lag24',
            'PM25_roll3', 'PM25_roll24', 'NOx', 'NOx_lag1', 'CO',
            'TEMP_MEAN', 'HUMIDITY', 'WSPD_SCLR', 'wind_u', 'wind_v',
            'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
            'is_rush_hour', 'is_weekend']

print(len(features), 'features')

## Train and test split

The split is by date and not random. A random split would let the model train on hours right next to the ones it is tested on, which makes the results look far better than they would be in real forecasting. Rows without a current PM2.5 reading or a target are dropped, since neither the baseline nor the models can be scored on them.

In [ ]:
model_df = df.dropna(subset=['PM25', 'target'])

train = model_df[model_df['DATE_PST'] < '2022-09-01']
test = model_df[model_df['DATE_PST'] >= '2022-09-01']

X_train, y_train = train[features], train['target']
X_test, y_test = test[features], test['target']

print('Train rows', len(train))
print('Test rows', len(test))

## Baseline

The simplest forecast says PM2.5 in six hours will match PM2.5 right now. Air quality changes slowly most of the time, so this persistence baseline is hard to beat, and any model worth using has to do better than it.

In [ ]:
def score(name, y_true, y_pred):
    return {
        'model': name,
        'MAE': round(mean_absolute_error(y_true, y_pred), 3),
        'RMSE': round(root_mean_squared_error(y_true, y_pred), 3),
        'R2': round(r2_score(y_true, y_pred), 3)
    }

results = [score('Persistence baseline', y_test, test['PM25'])]
pd.DataFrame(results)

## Models

The models predict how much PM2.5 will change over the next six hours, and that change is added to the current reading. Predicting the level directly fails badly for gradient boosting, since tree models can't predict values higher than anything in their training data. The highest training reading is about 163 µg/m³, and the smoke in October pushed readings past 340.

Ridge regression is a linear model, so it needs missing values filled and features scaled first. Histogram gradient boosting handles missing values on its own and can pick up nonlinear patterns.

In [ ]:
y_train_change = y_train - train['PM25']

ridge = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), Ridge(alpha=1.0))
ridge.fit(X_train, y_train_change)
ridge_pred = ridge.predict(X_test) + test['PM25']
results.append(score('Ridge regression', y_test, ridge_pred))

hgb = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=42)
hgb.fit(X_train, y_train_change)
hgb_pred = hgb.predict(X_test) + test['PM25']
results.append(score('Gradient boosting', y_test, hgb_pred))

results_df = pd.DataFrame(results)
results_df

## Results during the smoke

Scores across the whole test period can hide how a model does on the hours that matter most. This cell compares MAE on high PM2.5 hours, using 25 µg/m³ as the cutoff, which is the BC 24-hour air quality objective.

In [ ]:
smoke = test['PM25'] > 25
preds = {
    'Persistence baseline': test['PM25'],
    'Ridge regression': ridge_pred,
    'Gradient boosting': hgb_pred
}

print('High PM2.5 hours in test set', smoke.sum())
print()
for name, p in preds.items():
    mae_high = mean_absolute_error(y_test[smoke], p[smoke])
    mae_normal = mean_absolute_error(y_test[~smoke], p[~smoke])
    print(f'{name}  MAE high {mae_high:.2f}  MAE normal {mae_normal:.2f}')

In [ ]:
station = 'Vancouver Clark Drive'
mask = (test['STATION_NAME'] == station) & test['DATE_PST'].between('2022-10-10', '2022-10-25')
window = test[mask]

fig, ax = plt.subplots(figsize=(11, 4))
# x axis is the time being forecast, which is HORIZON hours after the row's timestamp
forecast_time = window['DATE_PST'] + pd.Timedelta(hours=HORIZON)
ax.plot(forecast_time, window['target'], label='actual', color='black')
ax.plot(forecast_time, preds['Gradient boosting'][mask], label='gradient boosting', alpha=0.8)
ax.plot(forecast_time, preds['Persistence baseline'][mask], label='persistence', alpha=0.6, linestyle='--')
ax.set_title(f'PM2.5 forecast {HORIZON} hours ahead at {station}, October 2022 smoke')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.legend()
plt.tight_layout()
plt.show()

## Which features matter

Permutation importance shuffles one feature at a time and measures how much worse the gradient boosting model gets at predicting the change. It runs on a sample of the test set to keep it quick.

In [ ]:
sample = test.sample(5000, random_state=42)
imp = permutation_importance(hgb, sample[features], sample['target'] - sample['PM25'],
                             n_repeats=5, random_state=42, scoring='neg_mean_absolute_error')

importance = pd.Series(imp.importances_mean, index=features).sort_values()

fig, ax = plt.subplots(figsize=(7, 6))
importance.plot.barh(ax=ax, color='steelblue')
ax.set_title('Permutation importance (increase in MAE)')
plt.tight_layout()
plt.show()

## How far ahead the models help

This cell reruns the baseline and gradient boosting model at several horizons. At one hour ahead the persistence baseline is hard to beat, and the models only pull ahead as the horizon gets longer.

In [ ]:
rows = []
for h in [1, 3, 6, 12, 24]:
    df['target_h'] = g['PM25'].shift(-h)
    m = df.dropna(subset=['PM25', 'target_h'])
    tr = m[m['DATE_PST'] < '2022-09-01']
    te = m[m['DATE_PST'] >= '2022-09-01']

    model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=42)
    model.fit(tr[features], tr['target_h'] - tr['PM25'])
    pred = model.predict(te[features]) + te['PM25']

    base_mae = mean_absolute_error(te['target_h'], te['PM25'])
    model_mae = mean_absolute_error(te['target_h'], pred)
    rows.append({'hours ahead': h,
                 'baseline MAE': round(base_mae, 2),
                 'model MAE': round(model_mae, 2),
                 'improvement %': round(100 * (base_mae - model_mae) / base_mae, 1)})

horizons = pd.DataFrame(rows)
horizons

## Findings

Write up what the results show here once you have run the notebook. Questions to answer

- How much do the models improve on the persistence baseline at six hours, overall and during smoke?
- At what horizon do the models start to beat persistence, and why might that be?
- Which features carry the most weight, and does anything besides recent PM2.5 matter?
- Where does the model fall behind, for example at the start of a smoke event?

## Next steps

- Add weather forecasts or smoke forecasts as features, since past readings can't see smoke coming
- Compare stations or regions to see where forecasting is hardest
- Try a model trained only on smoke season data